# Evaluación de Alineación Histológica: Pipeline VALIS+PhaseCorr y Pipeline GrandQC (ECC + Demons)
**Objetivo de este notebook:**
comparar pipeline (separación por Otsu + VALIS + métrica de phase correlation) contra otro pipeline (GrandQC + SIFT/RANSAC +micro-alineación ECC+ Demons), corriendo **ambos sobre los mismos TIF**, para responder
dos preguntas:

1. **¿Cuál método deja los tiles mejor alineados, en general?** (comparación
   agregada de métricas: distribución de desplazamiento residual en µm)
2. **¿Cómo se ve eso en el tejido real?** (mismo sector de tejido, resultado de
   tu pipeline vs el de ella, uno al lado del otro)

---
## CONFIGURACIÓN

**Objetivo:** Centralizar los parámetros del sistema, definir la resolución espacial del escáner (`MPP_BASE` = micrones por píxel) y validar la integridad de la estructura de archivos local previo a la ejecución.

In [ ]:
# Configuración de paths y parámetros para la evaluación propia de los modelos entrenados.
import random
import os

# Carpetas de pipeline de VALIS
CARPETA_BASE_VALIS = "C:/investigacion/tiles"
CARPETA_T1_VALIS = CARPETA_BASE_VALIS + "/muestra_1"   
CARPETA_T2_VALIS = CARPETA_BASE_VALIS + "/muestra_2"   
RUTA_CSV_VALIS = CARPETA_BASE_VALIS + "/indice_tiles_con_metricas.csv"

# Carpetas de pipeline GrandQC 
CARPETA_BASE_GRANDQC = "C:/Users/cande/OneDrive/Escritorio/GIA/datasetNefro"

CARPETA_T1_GRANDQC  = CARPETA_BASE_GRANDQC + "/ecc/tejido1"
CARPETA_T2_GRANDQC  = CARPETA_BASE_GRANDQC + "/ecc/tejido2"
RUTA_CSV_GRANDQC  = CARPETA_BASE_GRANDQC + "/manifest.csv"

# Micrones por pixel a nivel 0 - CONFIRMAR que sea el mismo valor que
# ingresaste manualmente cuando corriste la extracción de tiles.
MPP_BASE = 2.1665

CARPETA_SALIDA = "C:/Users/cande/OneDrive/Escritorio/GIA/comparacion_metodos"

# Semilla para reproducibilidad
SEMILLA = 42
random.seed(SEMILLA)
try:
    import numpy as _np
    _np.random.seed(SEMILLA)
except ImportError:
    pass

# Crear carpeta de salida si no existe
os.makedirs(CARPETA_SALIDA, exist_ok=True)

print("\nVerificando existencia de carpetas y archivos necesarios...")
for nombre, ruta in [("ecc/tejido1", CARPETA_T1_GRANDQC), ("ecc/tejido2", CARPETA_T2_GRANDQC),
                    ("manifest.csv", RUTA_CSV_GRANDQC)]:
    estado = "OK" if os.path.exists(ruta) else "No encontrado"
    print(f"[{estado}] {nombre} -> {ruta}")

print("\nSi ves algún [No encontrado] arriba, corregí CARPETA_BASE antes de seguir.")

Verificando existencia de carpetas y archivos necesarios...
[OK] ecc/tejido1 -> C:/Users/cande/OneDrive/Escritorio/GIA/datasetNefro/ecc/tejido1
[OK] ecc/tejido2 -> C:/Users/cande/OneDrive/Escritorio/GIA/datasetNefro/ecc/tejido2
[OK] manifest.csv -> C:/Users/cande/OneDrive/Escritorio/GIA/datasetNefro/manifest.csv

Si ves algún [No encontrado] arriba, corregí CARPETA_BASE antes de seguir.


In [ ]:
# Si tenés el metadata_pipeline.json generado por el Paso 1 de tu notebook
# (versión actualizada, la que guarda también los offsets de recorte), lo
# leemos acá para no tener que mantener MPP_BASE a mano en dos lugares y,
# sobre todo, para poder traducir coordenadas propias -> coordenadas del
# TIF original (necesario para la comparación visual de la Parte 3).
import json, os

RUTA_METADATA_PIPELINE = os.path.join(os.path.dirname(RUTA_CSV_VALIS), "..", "resultados", "metadata_pipeline.json")
RUTA_METADATA_PIPELINE = os.path.normpath(RUTA_METADATA_PIPELINE)

offset_muestra_1 = None
if os.path.exists(RUTA_METADATA_PIPELINE):
    with open(RUTA_METADATA_PIPELINE, "r") as f:
        _meta = json.load(f)
    MPP_BASE = _meta.get("mpp_base", MPP_BASE)
    offset_muestra_1 = _meta.get("offsets_recorte", {}).get("muestra_1")
    print(f"metadata_pipeline.json encontrado. mpp_base={MPP_BASE:.4f}")
    if offset_muestra_1:
        print(f"Offset de recorte de muestra_1: {offset_muestra_1}")
    else:
        print("No hay 'offsets_recorte' en el json todavía - actualizá el Paso 1 de tu "
            "notebook de alineación (la celda que guarda offsets_recorte) y volvé a correrlo, "
            "si no la Parte 3 (comparación visual por sector) no va a poder ubicar tus tiles "
            "en coordenadas del TIF original.")
else:
    print(f"No encontré {RUTA_METADATA_PIPELINE}. Ajustá la ruta a mano si hace falta, "
        "o segui con MPP_BASE hardcodeado arriba (la Parte 1 y 2 funcionan igual sin esto; "
        "solo la Parte 3 lo necesita).")

---
## Métrica Principal: phase correlation

**Objetivo:** Implementar la función de evaluación de desplazamiento residual basada en **Phase Correlation**. 

*   **Procesamiento:** Convierte las imágenes a escala de grises, aplica ecualización adaptativa de contraste (**CLAHE**) y extrae la respuesta de bordes mediante operadores **Sobel**.
*   **Independencia de Tinción:** Al evaluar la magnitud del gradiente (estructura anatómica) en lugar de la intensidad directa de píxeles, la métrica resulta invulnerable a variaciones de coloración entre cortes.
*   **Resultado:** Reporta el desplazamiento físico en micrones ($\mu m$) y la respuesta de correlación de fase.

In [ ]:
# Metódos de evaluación de tiles, para comparar los resultados de los modelos entrenados.
import cv2
import numpy as np
import pandas as pd
import os

# Configuración de tqdm para mostrar barra de progreso
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

_clahe_op = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def metrica_similitud_tiles(tile1, tile2, mpp=None, ventana=True,
                        modo="gradiente", usar_clahe=True):
    """
    Compara 2 tiles del mismo tamaño con phase correlation. Compara ESTRUCTURA
    (bordes, vía Sobel).

    mpp: micrones por pixel, para reportar el shift también en µm.
    Devuelve dict: shift_x_px, shift_y_px, shift_mag_px, phase_corr_response
    [, shift_mag_um].
    """
    g1 = cv2.cvtColor(tile1, cv2.COLOR_RGB2GRAY) if tile1.ndim == 3 else tile1.copy()
    g2 = cv2.cvtColor(tile2, cv2.COLOR_RGB2GRAY) if tile2.ndim == 3 else tile2.copy()

    if modo == "gradiente":
        if usar_clahe:
            g1 = _clahe_op.apply(g1)
            g2 = _clahe_op.apply(g2)
        gx1 = cv2.Sobel(g1, cv2.CV_32F, 1, 0, ksize=3)
        gy1 = cv2.Sobel(g1, cv2.CV_32F, 0, 1, ksize=3)
        f1 = cv2.magnitude(gx1, gy1)
        gx2 = cv2.Sobel(g2, cv2.CV_32F, 1, 0, ksize=3)
        gy2 = cv2.Sobel(g2, cv2.CV_32F, 0, 1, ksize=3)
        f2 = cv2.magnitude(gx2, gy2)
    elif modo == "crudo":
        f1, f2 = g1.astype(np.float32), g2.astype(np.float32)
    else:
        raise ValueError("modo debe ser 'gradiente' o 'crudo'")

    win = cv2.createHanningWindow((f1.shape[1], f1.shape[0]), cv2.CV_32F) if ventana else None
    (dx, dy), response = cv2.phaseCorrelate(f1, f2, win)
    shift_mag_px = float(np.hypot(dx, dy))

    resultado = {
        "shift_x_px": dx,
        "shift_y_px": dy,
        "shift_mag_px": shift_mag_px,
        "phase_corr_response": float(response),
    }
    if mpp is not None:
        resultado["shift_mag_um"] = shift_mag_px * mpp
    return resultado


def fraccion_tejido_de(img_rgb, umbral=220):
    """Fracción de píxeles que NO son fondo blanco."""
    gris = np.mean(img_rgb, axis=2) if img_rgb.ndim == 3 else img_rgb
    return float(np.mean(gris < umbral))


def leer_par_tiles(nombre, carpeta_1, carpeta_2):
    """Lee un par de tiles en RGB. Devuelve (None, None) si algo falta/está roto."""
    ruta_1 = os.path.join(carpeta_1, nombre)
    ruta_2 = os.path.join(carpeta_2, nombre)
    img1 = cv2.imread(ruta_1)
    img2 = cv2.imread(ruta_2)
    if img1 is None or img2 is None:
        return None, None
    return cv2.cvtColor(img1, cv2.COLOR_BGR2RGB), cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

print("\nEvaluación de tiles...")
print("Parte 1 lista: metrica_similitud_tiles, fraccion_tejido_de, leer_par_tiles")

Evaluación de tiles...
Parte 1 lista: metrica_similitud_tiles, fraccion_tejido_de, leer_par_tiles


---
## Ejecución del Registro de Métricas sobre Muestras Pareadas

**Objetivo:** Procesar en lote (*batch processing*) todas las imágenes pareadas pertenecientes a cada pipeline, calculando la alineación residual y la proporción de tejido efectivo (descartando fondo).

In [ ]:
def evaluar_pares_de_tiles(carpeta_1, carpeta_2, mpp, etiqueta_metodo,
                            modo="gradiente", verbose=True):
    """
    Corre metrica_similitud_tiles sobre TODOS los pares de tiles que comparten
    nombre de archivo entre carpeta_1 y carpeta_2.
    """
    if not os.path.isdir(carpeta_1) or not os.path.isdir(carpeta_2):
        if verbose:
            print(f"[{etiqueta_metodo}] no encontré {carpeta_1} o {carpeta_2}, se salta.")
        return pd.DataFrame()

    nombres_1 = set(os.listdir(carpeta_1))
    nombres_2 = set(os.listdir(carpeta_2))
    nombres_comunes = sorted(nombres_1 & nombres_2)

    if verbose:
        faltantes = (nombres_1 | nombres_2) - (nombres_1 & nombres_2)
        print(f"[{etiqueta_metodo}] {len(nombres_comunes)} pares encontrados"
            + (f" ({len(faltantes)} archivos sin pareja, se ignoran)" if faltantes else ""))

    resultados = []
    no_leidos = []
    for nombre in tqdm(nombres_comunes, desc=f"Métricas ({etiqueta_metodo})"):
        tile1, tile2 = leer_par_tiles(nombre, carpeta_1, carpeta_2)
        if tile1 is None:
            no_leidos.append(nombre)
            continue
        m = metrica_similitud_tiles(tile1, tile2, mpp=mpp, modo=modo)
        m["nombre"] = nombre
        m["metodo"] = etiqueta_metodo
        m["ancho_tile_px"] = tile1.shape[1]
        m["fraccion_tejido_t1"] = fraccion_tejido_de(tile1)
        m["fraccion_tejido_t2"] = fraccion_tejido_de(tile2)
        resultados.append(m)

    if no_leidos and verbose:
        print(f"  {len(no_leidos)} tiles no se pudieron leer y se saltearon.")

    return pd.DataFrame(resultados)

# Pipelines
df_valis = evaluar_pares_de_tiles(
    CARPETA_T1_VALIS, CARPETA_T2_VALIS, mpp=MPP_BASE,
    etiqueta_metodo="VALIS + PhaseCorr(gradiente)"
)

df_grandqc = evaluar_pares_de_tiles(
    CARPETA_T1_GRANDQC, CARPETA_T2_GRANDQC, mpp=MPP_BASE,
    etiqueta_metodo="GrandQC + ECC/Demons"
)

df_comparacion = pd.concat(
    [df_valis, df_grandqc],
    ignore_index=True
)

ruta_csv_comparacion = os.path.join(CARPETA_SALIDA, "metricas_comparacion.csv")
df_comparacion.to_csv(ruta_csv_comparacion, index=False)
print(f"\nGuardado: {ruta_csv_comparacion}  ({len(df_comparacion)} tiles en total)")

[GrandQC + ECC/Demons (alineación fina)] 1088 pares encontrados


Métricas (GrandQC + ECC/Demons (alineación fina)):   0%|          | 0/1088 [00:00<?, ?it/s]


Guardado: C:/Users/cande/OneDrive/Escritorio/GIA/datasetNefro/evaluacion_propia\metricas_propias_ecc.csv  (1088 tiles en total)


---
## Análisis Estadístico Descriptivo del Desplazamiento Residual

**Objetivo:** Cuantificar la precisión de la alineación en micrones ($\mu m$) a través de percentiles claves ($p_{50}$, $p_{75}$, $p_{90}$, $p_{95}$) y evaluar el cumplimiento bajo umbrales de tolerancia clínica (20 $\mu m$, 50 $\mu m$ y 100 $\mu m$).

*   **Representación Visual:** Generación de boxplots e histogramas de densidad para analizar la distribución del error residual.
*   **Significancia estadística:** Como los tiles de VALIS y los de GrandQC vienen de grillas de recorte independientes (no son el mismo tile en ambas pipelines), la comparación es entre **dos poblaciones**, no un par de mediciones apareadas. Por eso se suma un test de **Mann-Whitney U**, que no asume normalidad ni apareamiento, para saber si la diferencia entre métodos es estadísticamente significativa y no producto del azar/tamaño de muestra.


In [ ]:
# Analisis de resultados: percentiles, boxplot y histogramas
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# Resumen estadístico de shift_mag_um por método
resumen = (
    df_comparacion
    .groupby("metodo")["shift_mag_um"]
    .describe(percentiles=[.5, .75, .9, .95])
    .round(1)
)
print(resumen)

for umbral in (20, 50, 100):
    print(f"\n% de tiles con shift_mag_um <= {umbral}µm:")
    print(
        (df_comparacion.groupby("metodo")["shift_mag_um"]
        .apply(lambda s: (s <= umbral).mean() * 100)
        .round(1))
    )

# Test de Mann-Whitney U: compara las DOS poblaciones de shift_mag_um
# (no apareado, no asume distribución normal). H0: ambas vienen de la
# misma distribución. Si p < 0.05, la diferencia entre métodos es
# estadísticamente significativa (no es solo ruido de muestreo).
metodos = df_comparacion["metodo"].dropna().unique().tolist()
if len(metodos) == 2:
    grupo_a = df_comparacion.loc[df_comparacion["metodo"] == metodos[0], "shift_mag_um"].dropna()
    grupo_b = df_comparacion.loc[df_comparacion["metodo"] == metodos[1], "shift_mag_um"].dropna()
    stat, p_valor = mannwhitneyu(grupo_a, grupo_b, alternative="two-sided")
    print(f"\nMann-Whitney U ({metodos[0]} vs {metodos[1]}): U={stat:.1f}, p={p_valor:.2e}")
    if p_valor < 0.05:
        print("-> La diferencia entre métodos es estadísticamente significativa (p < 0.05).")
    else:
        print("-> No hay evidencia suficiente de diferencia entre métodos (p >= 0.05).")
else:
    print(f"\n(Se esperaban 2 métodos para el test de Mann-Whitney, hay {len(metodos)}; se omite.)")

print("\nGrafico de boxplot y histogramas de desplazamiento residual por método...")
# Boxplot
fig, ax = plt.subplots(figsize=(9, 5))
datos_box = [df_comparacion.loc[df_comparacion["metodo"] == m, "shift_mag_um"].dropna() for m in metodos]
ax.boxplot(datos_box, tick_labels=metodos, showfliers=False)
ax.set_ylabel("Desplazamiento residual (µm)")
ax.set_title("Calidad de alineación por método (menor = mejor)")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_SALIDA, "boxplot_comparacion.png"), dpi=150, bbox_inches="tight")
plt.show()

# Histogramas superpuestos
fig, ax = plt.subplots(figsize=(9, 5))
for m in metodos:
    sub = df_comparacion.loc[df_comparacion["metodo"] == m, "shift_mag_um"].dropna()
    ax.hist(sub, bins=40, alpha=0.5, label=f"{m} (n={len(sub)})", density=True)
ax.set_xlabel("Desplazamiento residual (µm)")
ax.set_ylabel("Densidad")
ax.set_title("Distribución de desplazamiento residual por método")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_SALIDA, "histograma_comparacion.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Validación Visual Localizada por Coordenadas Orgánicas

**Objetivo:** Mapear un punto de interés $(x, y)$ en espacio de coordenadas del portaobjetos original (Nivel 0) hacia el tile equivalente más cercano de **cada** pipeline, usando su respectivo índice (`manifest.csv` para GrandQC, `indice_tiles_con_metricas.csv` para VALIS).

*   **Utilidad:** Permite inspeccionar cualitativamente el acople de estructuras vasculares y glomerulares específicas, comparando la muestra de referencia (`tejido1`) contra la muestra transformada (`tejido2`) junto a su métrica puntual, para **las dos pipelines en simultáneo** (grilla 2x2: fila 1 = VALIS, fila 2 = GrandQC).
*   **Nota:** el manifest de GrandQC puede tener filas de `ecc` y de `geom` (si en algún momento volvés a generar esa carpeta); acá se filtra explícitamente por `carpeta == "ecc"`, que es lo único que se está evaluando en este notebook. También se filtra que el índice de VALIS tenga columna `nombre`, además de `x1_orig`/`y1_orig`, porque el resto del código busca el archivo por ese nombre.
*   Como las dos pipelines recortan tiles con grillas independientes, "el tile más cercano" no cae siempre en el mismo punto exacto — por eso se muestra la distancia real al punto pedido en el título de cada imagen, para saber qué tan aproximada es la comparación.


In [ ]:
# Comparación visual de tiles de un mismo sector del TIF original, entre referencia y alineado.

def indice_valis(ruta_csv):
    """Carga tu índice de tiles; falla con un mensaje claro si faltan
    x1_orig/y1_orig/nombre (agregalas primero corriendo la versión actualizada
    del Paso 1 + Paso 3 de tu notebook de alineación)."""
    df = pd.read_csv(ruta_csv)
    faltan = {"x1_orig", "y1_orig", "nombre"} - set(df.columns)
    if faltan:
        raise RuntimeError(
            f"Al índice le faltan columnas {faltan}. Actualizá el Paso 1 "
            "(guardado de offsets_recorte) y el Paso 3 (agregado de "
            "x1_orig/y1_orig) de tu notebook de alineación y volvé a "
            "correr la extracción de tiles."
        )
    return df


def indice_grandqc(ruta_manifest):
    """
    Carga mi manifest.csv y lo deja listo para buscar tiles por cercanía.
    Devuelve columnas: nombre, x1_orig, y1_orig, carpeta ('ecc').
    Se filtra a carpeta == 'ecc': es la única carpeta que evalúa este
    notebook (geom queda afuera de esta comparación).
    """
    df = pd.read_csv(ruta_manifest)
    df = df.rename(columns={"x1": "x1_orig", "y1": "y1_orig"})
    faltan = {"x1_orig", "y1_orig", "nombre", "carpeta"} - set(df.columns)
    if faltan:
        raise RuntimeError(f"Al manifest le faltan columnas {faltan}.")
    return df[df["carpeta"] == "ecc"].reset_index(drop=True)


def tile_mas_cercano(df_indice, x, y, radio_max=None):
    """Tile cuyo (x1_orig, y1_orig) está más cerca de (x, y). Devuelve
    también la distancia real, para poder mostrarla y saber qué tan
    aproximada es la comparación."""
    if df_indice.empty:
        return None, None
    d = np.hypot(df_indice["x1_orig"] - x, df_indice["y1_orig"] - y)
    i = d.idxmin()
    if radio_max is not None and d.loc[i] > radio_max:
        return None, None
    return df_indice.loc[i], float(d.loc[i])


df_indice_grandqc = indice_grandqc(RUTA_CSV_GRANDQC)
df_indice_valis = indice_valis(RUTA_CSV_VALIS)
print(f"Índice VALIS: {len(df_indice_valis)} tiles con coords originales")
print(f"Índice GRANDQC (solo carpeta ecc): {len(df_indice_grandqc)} tiles con coords originales")


def comparar_sector(x, y, radio_busqueda=1000, guardar=True):
    """
    Muestra, para el punto (x, y) del TIF original, el tile más cercano de
    CADA pipeline (VALIS arriba, GrandQC abajo): su referencia (tejido1) y
    su alineado (tejido2), lado a lado.
    """
    fila_valis, dist_valis = tile_mas_cercano(df_indice_valis, x, y, radio_max=radio_busqueda)
    fila_grandqc, dist_grandqc = tile_mas_cercano(df_indice_grandqc, x, y, radio_max=radio_busqueda)

    if fila_valis is None and fila_grandqc is None:
        print(f"No encontré tiles de ninguna de las dos pipelines cerca de ({x}, {y}).")
        return

    # Grilla 2x2: fila 0 = VALIS, fila 1 = GrandQC (antes esto usaba 1x2 y
    # la fila de GrandQC pisaba la de VALIS en los mismos ejes; separar en
    # 2 filas evita que una pipeline tape a la otra).
    fig, axes = plt.subplots(2, 2, figsize=(11, 11))

    if fila_valis is not None:
        t1 = cv2.cvtColor(cv2.imread(os.path.join(CARPETA_T1_VALIS, fila_valis["nombre"])), cv2.COLOR_BGR2RGB)
        t2 = cv2.cvtColor(cv2.imread(os.path.join(CARPETA_T2_VALIS, fila_valis["nombre"])), cv2.COLOR_BGR2RGB)
        m = metrica_similitud_tiles(t1, t2, mpp=MPP_BASE)
        axes[0][0].imshow(t1); axes[0][0].axis("off")
        axes[0][0].set_title(f"VALIS — Tejido 1\n{fila_valis['nombre']} (dist={dist_valis:.0f}px)", fontsize=9)
        axes[0][1].imshow(t2); axes[0][1].axis("off")
        axes[0][1].set_title(f"VALIS — Tejido 2\nshift={m['shift_mag_um']:.0f}µm", fontsize=9)
    else:
        for ax in axes[0]:
            ax.text(0.5, 0.5, "Sin tile VALIS cerca", ha="center", va="center"); ax.axis("off")

    if fila_grandqc is not None:
        t1 = cv2.cvtColor(cv2.imread(os.path.join(CARPETA_T1_GRANDQC, fila_grandqc["nombre"])), cv2.COLOR_BGR2RGB)
        t2 = cv2.cvtColor(cv2.imread(os.path.join(CARPETA_T2_GRANDQC, fila_grandqc["nombre"])), cv2.COLOR_BGR2RGB)
        m = metrica_similitud_tiles(t1, t2, mpp=MPP_BASE)
        axes[1][0].imshow(t1); axes[1][0].axis("off")
        axes[1][0].set_title(f"GrandQC — Tejido 1\n{fila_grandqc['nombre']} (dist={dist_grandqc:.0f}px)", fontsize=9)
        axes[1][1].imshow(t2); axes[1][1].axis("off")
        axes[1][1].set_title(f"GrandQC — Tejido 2\nshift={m['shift_mag_um']:.0f}µm", fontsize=9)
    else:
        for ax in axes[1]:
            ax.text(0.5, 0.5, "Sin tile GrandQC cerca", ha="center", va="center"); ax.axis("off")

    plt.suptitle(f"Mismo sector aproximado - TIF original en ({x}, {y})", fontsize=13)
    plt.tight_layout()
    if guardar:
        ruta = os.path.join(CARPETA_SALIDA, f"sector_{x}_{y}.png")
        plt.savefig(ruta, dpi=150, bbox_inches="tight")
        print(f"Guardado: {ruta}")
    plt.show()


# Sugerir sectores donde ambas pipelines tienen tiles cerca
def sugerir_sectores_en_comun(n=5, radio_busqueda=500):
    """
    Devuelve n coordenadas (x, y) donde df_indice_valis y df_indice_grandqc
    tienen tiles cerca uno del otro, para poder comparar visualmente.
    """
    candidatos = df_indice_valis.sample(min(len(df_indice_valis), 500), random_state=SEMILLA)
    sectores = []
    for _, fila_valis in candidatos.iterrows():
        x, y = fila_valis["x1_orig"], fila_valis["y1_orig"]
        fila_grandqc, _ = tile_mas_cercano(df_indice_grandqc, x, y, radio_max=radio_busqueda)
        if fila_grandqc is not None:
            sectores.append((x, y))
        if len(sectores) >= n:
            break
    return sectores

sectores_sugeridos = sugerir_sectores_en_comun(n=5)
print("Sectores sugeridos (x, y) donde ambas pipelines tienen tejido cerca:")
for x, y in sectores_sugeridos:
    print(f"\nComparando sector en ({x}, {y})...")
    comparar_sector(x=x, y=y)

# Ejemplo con input del usuario (opcional): dejá vacío + Enter para saltear.
# ejemplos que puedo probar: (10000, 10000), (15000, 8200)
x_usuario = input("Introduce la coordenada x del punto (Enter para saltear): ").strip()
if x_usuario:
    y_usuario = input("Introduce la coordenada y del punto: ").strip()
    x_usuario, y_usuario = float(x_usuario), float(y_usuario)
    print(f"Comparando sector en ({x_usuario}, {y_usuario})...")
    comparar_sector(x=x_usuario, y=y_usuario)
else:
    print("Se salteó la consulta manual de coordenadas.")

---
## Validación Cruzada mediante Información Mutua Normalizada (NMI)

**Objetivo:** Evaluar la fidelidad de la alineación utilizando una métrica de la teoría de la información (NMI) independiente de la correlación de fase.

*   **Fundamento:** La NMI mide la entropía conjunta entre ambas imágenes. Valores más altos indican una mayor coincidencia en la distribución espacial de las estructuras, siendo idónea para cortes histológicos con diferentes tinciones.

In [ ]:
def informacion_mutua_normalizada(img1_gray, img2_gray, bins=32):
    """
    NMI: familia teórico-informacional, NO de correlación.
    Rango aproximado [0, 1]; más alto = mayor correspondencia estructural.
    """
    hist_2d, _, _ = np.histogram2d(img1_gray.ravel(), img2_gray.ravel(), bins=bins)
    pxy = hist_2d / float(np.sum(hist_2d))
    px = np.sum(pxy, axis=1)
    py = np.sum(pxy, axis=0)
    px_py = px[:, None] * py[None, :]
    nzs = pxy > 0
    mi = np.sum(pxy[nzs] * np.log(pxy[nzs] / px_py[nzs]))
    hx = -np.sum(px[px > 0] * np.log(px[px > 0]))
    hy = -np.sum(py[py > 0] * np.log(py[py > 0]))
    if (hx + hy) == 0:
        return 0.0
    return float(2.0 * mi / (hx + hy))


def metrica_nmi_tiles(tile1_rgb, tile2_rgb, bins_nmi=32):
    """Devuelve el NMI entre dos tiles."""
    g1 = cv2.cvtColor(tile1_rgb, cv2.COLOR_RGB2GRAY) if tile1_rgb.ndim == 3 else tile1_rgb.copy()
    g2 = cv2.cvtColor(tile2_rgb, cv2.COLOR_RGB2GRAY) if tile2_rgb.ndim == 3 else tile2_rgb.copy()
    return {"nmi": informacion_mutua_normalizada(g1, g2, bins=bins_nmi)}


def evaluar_pares_nmi(carpeta_1, carpeta_2, etiqueta_metodo, verbose=True):
    if not os.path.isdir(carpeta_1) or not os.path.isdir(carpeta_2):
        if verbose:
            print(f"[{etiqueta_metodo}] no encontré {carpeta_1} o {carpeta_2}, se salta.")
        return pd.DataFrame()

    nombres_1 = set(os.listdir(carpeta_1))
    nombres_2 = set(os.listdir(carpeta_2))
    nombres_comunes = sorted(nombres_1 & nombres_2)

    if verbose:
        print(f"[{etiqueta_metodo}] {len(nombres_comunes)} pares a evaluar con NMI")

    resultados = []
    for nombre in tqdm(nombres_comunes, desc=f"NMI ({etiqueta_metodo})"):
        tile1, tile2 = leer_par_tiles(nombre, carpeta_1, carpeta_2)
        if tile1 is None:
            continue
        m = metrica_nmi_tiles(tile1, tile2)
        m["nombre"] = nombre
        m["metodo"] = etiqueta_metodo
        resultados.append(m)

    return pd.DataFrame(resultados)


# Métrica NMI para los tiles (GrandQC)
df_nmi_grandqc = evaluar_pares_nmi(
    CARPETA_T1_GRANDQC, CARPETA_T2_GRANDQC,
    etiqueta_metodo="GrandQC + ECC/Demons"
)

# Métrica NMI para los tiles (VALIS)
df_nmi_valis = evaluar_pares_nmi(
    CARPETA_T1_VALIS, CARPETA_T2_VALIS,
    etiqueta_metodo="VALIS"
)

# Guardar resultados de NMI en CSV
df_nmi = pd.concat([df_nmi_grandqc, df_nmi_valis], ignore_index=True)

ruta_csv_indep = os.path.join(CARPETA_SALIDA, "metrica_nmi_comparativa.csv")
df_nmi.to_csv(ruta_csv_indep, index=False)
print(f"\nGuardado: {ruta_csv_indep}  ({len(df_nmi)} tiles analizados en total)")

---
## Distribución Estadística de la Métrica NMI

**Objetivo:** Consolidar las estadísticas descriptivas y la representación gráfica (boxplot) de la Información Mutua Normalizada para el conjunto de datos procesado.

In [ ]:
# Resumen NMI
print("Información Mutua Normalizada (NMI)")
print(df_nmi.groupby("metodo")["nmi"].describe(percentiles=[.1, .5, .9]).round(3))

fig, ax = plt.subplots(figsize=(7, 5))
metodos = df_nmi["metodo"].dropna().unique().tolist()
datos_nmi = [df_nmi.loc[df_nmi["metodo"] == m, "nmi"].dropna() for m in metodos]
ax.boxplot(datos_nmi, tick_labels=metodos, showfliers=False)
ax.set_ylabel("NMI (más alto = mejor)")
ax.set_title("Información Mutua Normalizada[NMI] - GrandQC vs VALIS")
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_SALIDA, "nmi_comparativa.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## Análisis de Concordancia entre Métricas

**Objetivo:** Validar la coherencia matemática entre el desplazamiento residual ($\mu m$) y la Información Mutua Normalizada (NMI) mediante el coeficiente de correlación de rangos de Spearman ($\rho$).

*   **Criterio de Validación:** Dado que el desplazamiento mide *error* (menor es mejor) y el NMI mide *coincidencia* (mayor es mejor), se busca confirmar una **correlación negativa significativa** ($\rho < -0.40$). Esto demuestra que ambas métricas independientes convergen en el mismo diagnóstico de calidad de alineación.

In [ ]:
# Cruce de métricas. 
# Unimos los resultados de la evaluación de tiles con las métricas de phase correlation y NMI.
df_cruce = df_comparacion.merge(
    df_nmi[["nombre", "metodo", "nmi"]],
    on=["nombre", "metodo"], how="inner", suffixes=("", "_indep")
)

print(f"Tiles con las 2 métricas cruzadas: {len(df_cruce)}")

# Correlación de Spearman (monótona, no asume linealidad) entre
# phase-correlation y NMI
from scipy.stats import spearmanr

rho_nmi, p_nmi = spearmanr(df_cruce["shift_mag_um"], df_cruce["nmi"])
print(f"\nCorrelación GLOBAL phase_corr vs NMI: rho={rho_nmi:.3f} (p={p_nmi:.1e})")

# Correlación por método
print("\nDesglose de correlación por método:")
for m in df_cruce["metodo"].unique():
    sub = df_cruce[df_cruce["metodo"] == m]
    rho, p = spearmanr(sub["shift_mag_um"], sub["nmi"])
    print(f" - {m}: rho={rho:.3f} (p={p:.1e})")

# Scatter para ver visualmente el acuerdo/desacuerdo
fig, ax = plt.subplots(figsize=(7, 6))
metodos = df_cruce["metodo"].dropna().unique().tolist()

# Scatter plot de shift_mag_um vs nmi, coloreado por método
for m in metodos:
    sub = df_cruce[df_cruce["metodo"] == m]
    ax.scatter(sub["shift_mag_um"], sub["nmi"], alpha=0.4, s=12, label=m)
ax.set_xlabel("Desplazamiento residual (µm)")
ax.set_ylabel("Información Mutua Normalizada (NMI)")
ax.set_title("Concordancia entre Correlación de Fase y NMI")
ax.legend(fontsize=9, title="Métodos")

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_SALIDA, "scatter_phasecorr_vs_nmi_comparativa.png"), dpi=150, bbox_inches="tight")
plt.show()